# Fixed train/validation split + memory-safe experiment pipeline for multi-label task

Этот ноутбук делает две ключевые вещи по best practices:

1. **Один раз** создает и сохраняет фиксированный `train/validation` split для `main`, `extra` и `target`, чтобы вся команда сравнивала эксперименты на **одном и том же** validation.
2. Дает **гибкий memory-safe пайплайн** для всех EDA/feature экспериментов:
   - `main only` / `main + all extra` / `main + часть extra`
   - `extra`: `none` / `PCA` / `SVD`
   - стратегии фильтрации фичей
   - стратегии заполнения пропусков
   - запуск сетки экспериментов с логированием метрик в таблицу и, при желании, в MLflow

Ниже код ориентирован на:
- **multi-label** split без утечки;
- fit всех селекторов/импьютеров/редьюсеров **только на train части**;
- минимизацию проблем с памятью за счет:
  - lazy чтения Parquet через `polars.scan_parquet`
  - чтения только нужных колонок extra
  - downcast в `float32`
  - sampling для тяжелых EDA шагов (например, корреляции)
  - аккуратного `gc.collect()` между экспериментами

> Важно: сначала выполните ячейку **"CREATE AND SAVE FIXED SPLIT FILES"**, а уже потом ячейку **"RUN ALL EDA EXPERIMENTS"**.


In [ ]:
# Если нужно:
# !pip install polars pandas pyarrow scikit-learn catboost iterative-stratification mlflow

import os
import gc
import json
import math
import warnings
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Iterable

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import polars as pl

from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from sklearn.model_selection import StratifiedShuffleSplit

from catboost import CatBoostClassifier, Pool, CatBoostError

try:
    from iterstrat.ml_stratifiers import (
        MultilabelStratifiedShuffleSplit,
        MultilabelStratifiedKFold,
    )
    HAS_ITERSTRAT = True
except Exception:
    HAS_ITERSTRAT = False

try:
    import mlflow
    import mlflow.catboost
    HAS_MLFLOW = True
except Exception:
    HAS_MLFLOW = False

print("HAS_ITERSTRAT =", HAS_ITERSTRAT)
print("HAS_MLFLOW    =", HAS_MLFLOW)


In [ ]:
# ============================================================
# CONFIG
# ============================================================

@dataclass
class PathsConfig:
    data_dir: str = "data"
    split_dir: str = "prepared_split"

    train_main_name: str = "train_main_features.parquet"
    test_main_name: str = "test_main_features.parquet"
    train_extra_name: str = "train_extra_features.parquet"
    test_extra_name: str = "test_extra_features.parquet"
    train_target_name: str = "train_target.parquet"
    sample_submit_name: str = "sample_submit.parquet"

    @property
    def train_main_path(self) -> str:
        return str(Path(self.data_dir) / self.train_main_name)

    @property
    def test_main_path(self) -> str:
        return str(Path(self.data_dir) / self.test_main_name)

    @property
    def train_extra_path(self) -> str:
        return str(Path(self.data_dir) / self.train_extra_name)

    @property
    def test_extra_path(self) -> str:
        return str(Path(self.data_dir) / self.test_extra_name)

    @property
    def train_target_path(self) -> str:
        return str(Path(self.data_dir) / self.train_target_name)

    @property
    def sample_submit_path(self) -> str:
        return str(Path(self.data_dir) / self.sample_submit_name)


@dataclass
class SplitConfig:
    val_size: float = 0.20
    random_state: int = 42
    prefer_multilabel_shuffle: bool = True
    fallback_kfold_splits: int = 5  # используется только если нет shuffle split
    save_extra_split_files: bool = True


@dataclass
class ExperimentConfig:
    # reproducibility
    random_state: int = 42

    # data usage
    use_extra_mode: str = "none"  
    # options: "none", "all", "topk", "explicit"
    extra_top_k: Optional[int] = None
    extra_explicit_cols: Optional[List[str]] = None

    # extra column selection before reduction
    extra_filter_strategy: str = "score_topk"
    # options: "none", "missing_only", "variance_only", "missing_and_variance", "score_topk"
    drop_extra_missing_over: Optional[float] = 0.98
    variance_threshold: float = 0.0
    max_extra_keep_after_filter: Optional[int] = 1000

    # extra reduction
    extra_reduce_method: str = "none"
    # options: "none", "pca", "svd"
    extra_reduce_dim: Optional[int] = None

    # feature filtering on merged train-only statistics
    feature_filter_strategy: str = "drop_constant_high_missing"
    # options:
    # "none"
    # "drop_constant"
    # "drop_high_missing"
    # "drop_constant_high_missing"
    # "drop_constant_high_missing_corr"
    feature_missing_threshold: float = 0.98
    corr_threshold: float = 0.995
    corr_sample_rows: int = 30000

    # missing values
    missing_strategy: str = "native_catboost"
    # options:
    # "native_catboost", "median", "zero", "median_plus_ind", "zero_plus_ind"
    indicator_missing_threshold: float = 0.95

    # optional subsampling for quick debug only
    sample_rows: Optional[int] = None

    # CatBoost
    iterations: int = 2000
    learning_rate: float = 0.05
    depth: int = 6
    l2_leaf_reg: float = 5.0
    early_stopping_rounds: int = 150
    verbose_eval: int = 200

    task_type: str = "GPU"
    devices: str = "0"
    fallback_to_cpu_on_oom: bool = True
    gpu_ram_part: float = 0.80
    gpu_cat_features_storage: str = "CpuPinnedMemory"
    max_ctr_complexity: int = 1
    border_count: int = 128
    thread_count: int = -1

    # logging
    enable_mlflow: bool = False
    mlflow_tracking_uri: str = "http://localhost:5000"
    mlflow_experiment_name: str = "cyberpolka_fixed_val_experiments"


PATHS = PathsConfig()
SPLIT_CFG = SplitConfig()


In [ ]:
# ============================================================
# UTILS
# ============================================================

def ensure_dir(path: str) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)

def print_header(text: str) -> None:
    print("\n" + "=" * 100)
    print(text)
    print("=" * 100)

def read_parquet_pl(path: str, columns: Optional[List[str]] = None) -> pl.DataFrame:
    if columns is None:
        return pl.read_parquet(path)
    return pl.scan_parquet(path).select(columns).collect()

def get_target_cols(columns: Iterable[str]) -> List[str]:
    return [c for c in columns if c.startswith("target")]

def get_cat_cols(columns: Iterable[str]) -> List[str]:
    return [c for c in columns if c.startswith("cat_feature")]

def get_num_cols(columns: Iterable[str]) -> List[str]:
    return [c for c in columns if c.startswith("num_feature") or c.startswith("extra_")]

def reduce_mem_float32(df: pd.DataFrame, exclude_cols: Optional[List[str]] = None) -> pd.DataFrame:
    exclude_cols = set(exclude_cols or [])
    for col in df.columns:
        if col in exclude_cols:
            continue
        if pd.api.types.is_float64_dtype(df[col]):
            df[col] = df[col].astype(np.float32)
        elif pd.api.types.is_int64_dtype(df[col]):
            col_min = df[col].min()
            col_max = df[col].max()
            if col_min >= np.iinfo(np.int32).min and col_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
    return df

def cast_cat_features_pl(df: pl.DataFrame, cat_cols: List[str], as_string: bool = True) -> pl.DataFrame:
    if not cat_cols:
        return df
    exprs = []
    for c in cat_cols:
        if as_string:
            exprs.append(
                pl.when(pl.col(c).is_null())
                .then(pl.lit("__MISSING__"))
                .otherwise(pl.col(c).cast(pl.Int64, strict=False).cast(pl.Utf8))
                .alias(c)
            )
        else:
            exprs.append(pl.col(c).cast(pl.Int32, strict=False).alias(c))
    return df.with_columns(exprs)

def maybe_subsample_by_split(
    train_df: pd.DataFrame,
    target_df: pd.DataFrame,
    sample_rows: Optional[int],
    random_state: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if sample_rows is None or sample_rows >= len(train_df):
        return train_df, target_df
    rng = np.random.RandomState(random_state)
    idx = np.sort(rng.choice(len(train_df), size=sample_rows, replace=False))
    return (
        train_df.iloc[idx].reset_index(drop=True),
        target_df.iloc[idx].reset_index(drop=True),
    )

def safe_mlflow_setup(cfg: ExperimentConfig):
    if cfg.enable_mlflow and HAS_MLFLOW:
        mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
        mlflow.set_experiment(cfg.mlflow_experiment_name)

def safe_mlflow_log_params(params: Dict):
    if HAS_MLFLOW:
        for k, v in params.items():
            if isinstance(v, (list, tuple, dict)):
                mlflow.log_param(k, json.dumps(v, ensure_ascii=False))
            else:
                mlflow.log_param(k, v)

def build_catboost_params(cfg: ExperimentConfig, random_seed: int, with_eval_metric: bool = True) -> Dict:
    params = {
        "iterations": cfg.iterations,
        "learning_rate": cfg.learning_rate,
        "depth": cfg.depth,
        "l2_leaf_reg": cfg.l2_leaf_reg,
        "loss_function": "MultiLogloss",
        "random_seed": random_seed,
        "nan_mode": "Min",
        "verbose": cfg.verbose_eval,
        "task_type": cfg.task_type,
        "thread_count": cfg.thread_count,
        "border_count": cfg.border_count,
        "max_ctr_complexity": cfg.max_ctr_complexity,
        "allow_writing_files": False,
    }
    if with_eval_metric:
        params["eval_metric"] = "MultiLogloss"
    if cfg.task_type == "GPU":
        params["devices"] = cfg.devices
        params["gpu_ram_part"] = cfg.gpu_ram_part
        params["gpu_cat_features_storage"] = cfg.gpu_cat_features_storage
    return params

def fit_catboost_model(
    train_pool: Pool,
    valid_pool: Optional[Pool],
    cfg: ExperimentConfig,
    random_seed: int,
    use_best_model: bool = True,
):
    params = build_catboost_params(cfg=cfg, random_seed=random_seed, with_eval_metric=(valid_pool is not None))
    model = CatBoostClassifier(**params)

    fit_kwargs = {}
    if valid_pool is not None:
        fit_kwargs["eval_set"] = valid_pool
        fit_kwargs["use_best_model"] = use_best_model
        fit_kwargs["early_stopping_rounds"] = cfg.early_stopping_rounds

    try:
        model.fit(train_pool, **fit_kwargs)
        return model
    except CatBoostError as e:
        msg = str(e).lower()
        is_gpu_oom = cfg.task_type == "GPU" and any(x in msg for x in ["bad allocation", "out of memory", "cuda"])
        if not (cfg.fallback_to_cpu_on_oom and is_gpu_oom):
            raise

        print_header("GPU OOM DETECTED -> RETRY ON CPU")

        cpu_params = build_catboost_params(cfg=cfg, random_seed=random_seed, with_eval_metric=(valid_pool is not None))
        cpu_params["task_type"] = "CPU"
        cpu_params.pop("devices", None)
        cpu_params.pop("gpu_ram_part", None)
        cpu_params.pop("gpu_cat_features_storage", None)

        cpu_model = CatBoostClassifier(**cpu_params)
        cpu_model.fit(train_pool, **fit_kwargs)
        return cpu_model

def multilabel_metrics(y_true: np.ndarray, y_prob: np.ndarray, eps: float = 1e-7) -> Dict[str, float]:
    n_targets = y_true.shape[1]
    aucs, pr_aucs, bces = [], [], []

    for j in range(n_targets):
        yt = y_true[:, j]
        yp = np.clip(y_prob[:, j], eps, 1.0 - eps)

        if len(np.unique(yt)) > 1:
            aucs.append(roc_auc_score(yt, yp))
            pr_aucs.append(average_precision_score(yt, yp))
        else:
            aucs.append(np.nan)
            pr_aucs.append(np.nan)

        bces.append(log_loss(yt, yp, labels=[0, 1]))

    row_top1_hit = np.mean(
        (y_true[np.arange(len(y_true)), np.argmax(y_prob, axis=1)] == 1).astype(np.float32)
    )

    return {
        "macro_auc": float(np.nanmean(aucs)),
        "macro_pr_auc": float(np.nanmean(pr_aucs)),
        "mean_bce": float(np.mean(bces)),
        "hit_at_1": float(row_top1_hit),
    }


## CREATE AND SAVE FIXED SPLIT FILES

Эта ячейка должна запускаться **один раз на проект**.  
Она:

- строит **фиксированный multi-label split**
- сохраняет:
  - `split_manifest.parquet`
  - `train_main_train.parquet`
  - `train_main_val.parquet`
  - `train_target_train.parquet`
  - `train_target_val.parquet`
  - `train_extra_train.parquet`
  - `train_extra_val.parquet`

Так вся команда работает с **одними и теми же** train/validation файлами, а не каждый раз генерирует split заново.


In [ ]:
# ============================================================
# FIXED MULTI-LABEL SPLIT CREATION
# ============================================================

def choose_train_val_indices_multilabel(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    split_cfg: SplitConfig,
) -> Tuple[np.ndarray, np.ndarray, str]:
    y_values = y_df.values

    # Лучший вариант: multilabel shuffle split
    if HAS_ITERSTRAT and split_cfg.prefer_multilabel_shuffle:
        msss = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=split_cfg.val_size,
            random_state=split_cfg.random_state,
        )
        tr_idx, va_idx = next(msss.split(X_df, y_values))
        return tr_idx, va_idx, "MultilabelStratifiedShuffleSplit"

    # Если shuffle split недоступен, используем multilabel k-fold и берем первый fold
    if HAS_ITERSTRAT:
        n_splits = split_cfg.fallback_kfold_splits
        mskf = MultilabelStratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=split_cfg.random_state,
        )
        tr_idx, va_idx = next(mskf.split(X_df, y_values))
        return tr_idx, va_idx, f"MultilabelStratifiedKFold(first_fold_of_{n_splits})"

    # Fallback: стратификация по числу активных таргетов в строке
    row_sum = y_df.sum(axis=1).values
    sss = StratifiedShuffleSplit(
        n_splits=1,
        test_size=split_cfg.val_size,
        random_state=split_cfg.random_state,
    )
    tr_idx, va_idx = next(sss.split(X_df, row_sum))
    return tr_idx, va_idx, "StratifiedShuffleSplit(row_sum_fallback)"


def report_split_quality(y_train: pd.DataFrame, y_val: pd.DataFrame) -> pd.DataFrame:
    rep = pd.DataFrame({
        "target": y_train.columns,
        "train_rate": y_train.mean().values,
        "val_rate": y_val.mean().values,
    })
    rep["abs_diff"] = (rep["train_rate"] - rep["val_rate"]).abs()
    return rep.sort_values("abs_diff", ascending=False)


def materialize_fixed_split(paths: PathsConfig, split_cfg: SplitConfig) -> Dict:
    ensure_dir(paths.split_dir)

    print_header("LOAD RAW DATA FOR SPLIT")
    train_main = pl.read_parquet(paths.train_main_path)
    train_target = pl.read_parquet(paths.train_target_path)

    if "customer_id" not in train_main.columns or "customer_id" not in train_target.columns:
        raise ValueError("В train_main и train_target должен быть customer_id")

    # Приведем к общему пересечению ID и одинаковому порядку
    common_ids = (
        train_main.select("customer_id")
        .join(train_target.select("customer_id"), on="customer_id", how="inner")
        .sort("customer_id")
    )

    train_main = common_ids.join(train_main, on="customer_id", how="left")
    train_target = common_ids.join(train_target, on="customer_id", how="left")

    target_cols = get_target_cols(train_target.columns)
    if not target_cols:
        raise ValueError("Не найдены target колонки с префиксом 'target'")

    X_stub = train_main.select(["customer_id"]).to_pandas()
    y_pd = train_target.select(target_cols).to_pandas().astype(np.int8)

    tr_idx, va_idx, split_method = choose_train_val_indices_multilabel(X_stub, y_pd, split_cfg)

    split_manifest = pd.DataFrame({
        "customer_id": common_ids["customer_id"].to_list(),
        "split": np.where(np.isin(np.arange(len(common_ids)), va_idx), "val", "train"),
    })

    split_manifest_pl = pl.from_pandas(split_manifest)
    split_manifest_path = str(Path(paths.split_dir) / "split_manifest.parquet")
    split_manifest_pl.write_parquet(split_manifest_path)

    train_ids = split_manifest.loc[split_manifest["split"] == "train", "customer_id"].tolist()
    val_ids = split_manifest.loc[split_manifest["split"] == "val", "customer_id"].tolist()

    train_ids_pl = pl.DataFrame({"customer_id": train_ids})
    val_ids_pl = pl.DataFrame({"customer_id": val_ids})

    # materialize main
    print_header("SAVE SPLIT FILES: MAIN + TARGET")
    train_main_train = train_main.join(train_ids_pl, on="customer_id", how="inner")
    train_main_val = train_main.join(val_ids_pl, on="customer_id", how="inner")

    train_target_train = train_target.join(train_ids_pl, on="customer_id", how="inner")
    train_target_val = train_target.join(val_ids_pl, on="customer_id", how="inner")

    train_main_train.write_parquet(str(Path(paths.split_dir) / "train_main_train.parquet"))
    train_main_val.write_parquet(str(Path(paths.split_dir) / "train_main_val.parquet"))
    train_target_train.write_parquet(str(Path(paths.split_dir) / "train_target_train.parquet"))
    train_target_val.write_parquet(str(Path(paths.split_dir) / "train_target_val.parquet"))

    split_quality = report_split_quality(
        train_target_train.select(target_cols).to_pandas(),
        train_target_val.select(target_cols).to_pandas(),
    )
    split_quality.to_csv(str(Path(paths.split_dir) / "split_quality_report.csv"), index=False)

    # materialize extra only once as well
    if split_cfg.save_extra_split_files and os.path.exists(paths.train_extra_path):
        print_header("SAVE SPLIT FILES: EXTRA")
        train_extra = pl.read_parquet(paths.train_extra_path)
        train_extra = common_ids.join(train_extra, on="customer_id", how="left")

        train_extra_train = train_extra.join(train_ids_pl, on="customer_id", how="inner")
        train_extra_val = train_extra.join(val_ids_pl, on="customer_id", how="inner")

        train_extra_train.write_parquet(str(Path(paths.split_dir) / "train_extra_train.parquet"))
        train_extra_val.write_parquet(str(Path(paths.split_dir) / "train_extra_val.parquet"))

    metadata = {
        "random_state": split_cfg.random_state,
        "val_size": split_cfg.val_size,
        "split_method": split_method,
        "n_total": int(len(common_ids)),
        "n_train": int(len(train_ids)),
        "n_val": int(len(val_ids)),
        "n_targets": int(len(target_cols)),
        "target_cols": target_cols,
    }
    with open(Path(paths.split_dir) / "split_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print_header("SPLIT DONE")
    print(json.dumps(metadata, ensure_ascii=False, indent=2))
    print("\nTop split target diffs:")
    print(split_quality.head(10).to_string(index=False))

    return metadata


# RUN ONCE:
split_metadata = materialize_fixed_split(PATHS, SPLIT_CFG)


## Pipeline helpers for experiments

Ниже — основной пайплайн.  
Ключевые best practices, которые соблюдены:

- split уже фиксирован и не меняется между экспериментами;
- любые selectors / imputers / reducers fit'ятся **только на train**;
- extra читается **только по нужным колонкам**;
- для тяжелых шагов используются **sampling** и `float32`;
- все эксперименты можно гонять из одной ячейки, меняя только конфиг/сетку.


In [ ]:
# ============================================================
# LOAD PREPARED SPLIT FILES + EDA HELPERS
# ============================================================

def load_split_parts(paths: PathsConfig):
    train_main_train = pl.read_parquet(Path(paths.split_dir) / "train_main_train.parquet")
    train_main_val = pl.read_parquet(Path(paths.split_dir) / "train_main_val.parquet")
    train_target_train = pl.read_parquet(Path(paths.split_dir) / "train_target_train.parquet")
    train_target_val = pl.read_parquet(Path(paths.split_dir) / "train_target_val.parquet")

    train_extra_train_path = Path(paths.split_dir) / "train_extra_train.parquet"
    train_extra_val_path = Path(paths.split_dir) / "train_extra_val.parquet"

    has_extra = train_extra_train_path.exists() and train_extra_val_path.exists()

    return {
        "train_main_train": train_main_train,
        "train_main_val": train_main_val,
        "train_target_train": train_target_train,
        "train_target_val": train_target_val,
        "train_extra_train_path": str(train_extra_train_path) if has_extra else None,
        "train_extra_val_path": str(train_extra_val_path) if has_extra else None,
    }


def quick_missing_report_pl(df: pl.DataFrame, name: str, top_n: int = 30) -> pd.DataFrame:
    rep = (
        df.null_count()
        .unpivot(variable_name="column", value_name="missing_count")
        .with_columns((pl.col("missing_count") / df.height).alias("missing_rate"))
        .sort("missing_count", descending=True)
    )
    out = rep.to_pandas()
    print_header(f"MISSING REPORT: {name}")
    print(out.head(top_n).to_string(index=False))
    return out


def quick_target_eda(target_pl: pl.DataFrame) -> pd.DataFrame:
    target_cols = get_target_cols(target_pl.columns)
    target_pd = target_pl.select(target_cols).to_pandas()
    row_sum = target_pd.sum(axis=1)
    means = target_pd.mean().sort_values(ascending=False)

    print_header("TARGET EDA")
    print("n_targets:", len(target_cols))
    print("\nTop target prevalences:")
    print(means.head(15).to_string())
    print("\nRow sum stats:")
    print(row_sum.describe())

    return pd.DataFrame({
        "target": means.index,
        "prevalence": means.values,
    })


def get_extra_columns_from_split(paths: PathsConfig) -> List[str]:
    extra_path = Path(paths.split_dir) / "train_extra_train.parquet"
    if not extra_path.exists():
        return []
    schema = pl.scan_parquet(str(extra_path)).collect_schema().names()
    return [c for c in schema if c != "customer_id"]


def compute_extra_stats(extra_train_path: str) -> pd.DataFrame:
    # легкий профайл колонок extra на TRAIN части split
    schema = pl.scan_parquet(extra_train_path).collect_schema().names()
    extra_cols = [c for c in schema if c != "customer_id"]

    if not extra_cols:
        return pd.DataFrame(columns=["column", "missing_rate", "mean", "std", "n_unique_approx"])

    lazy = pl.scan_parquet(extra_train_path)
    stats_frames = []

    # батчами, чтобы не раздувать память и не строить гигантское выражение на тысячи колонок сразу
    batch_size = 200
    for i in range(0, len(extra_cols), batch_size):
        batch = extra_cols[i:i+batch_size]
        exprs = []
        for c in batch:
            exprs.extend([
                pl.col(c).is_null().mean().alias(f"{c}__missing_rate"),
                pl.col(c).cast(pl.Float64, strict=False).mean().alias(f"{c}__mean"),
                pl.col(c).cast(pl.Float64, strict=False).std().alias(f"{c}__std"),
                pl.col(c).approx_n_unique().alias(f"{c}__n_unique_approx"),
            ])
        stats_frames.append(lazy.select(exprs).collect())

    merged = {}
    for chunk_df in stats_frames:
        merged.update(chunk_df.to_dicts()[0])

    rows = []
    for c in extra_cols:
        rows.append({
            "column": c,
            "missing_rate": merged.get(f"{c}__missing_rate"),
            "mean": merged.get(f"{c}__mean"),
            "std": merged.get(f"{c}__std"),
            "n_unique_approx": merged.get(f"{c}__n_unique_approx"),
        })

    out = pd.DataFrame(rows)
    out["std"] = out["std"].fillna(0.0)
    out["quality_score"] = (1.0 - out["missing_rate"].fillna(1.0)) * np.log1p(out["std"].clip(lower=0.0))
    out = out.sort_values(["quality_score", "std"], ascending=False).reset_index(drop=True)
    return out


def load_extra_subset(extra_path: str, selected_cols: List[str]) -> pl.DataFrame:
    cols = ["customer_id"] + selected_cols
    return pl.scan_parquet(extra_path).select(cols).collect()


def choose_extra_columns(
    extra_stats: pd.DataFrame,
    cfg: ExperimentConfig,
) -> List[str]:
    if cfg.use_extra_mode == "none":
        return []

    all_cols = extra_stats["column"].tolist()
    if cfg.use_extra_mode == "all":
        base_cols = all_cols
    elif cfg.use_extra_mode == "topk":
        if cfg.extra_top_k is None:
            raise ValueError("Для use_extra_mode='topk' нужно задать extra_top_k")
        base_cols = extra_stats.head(cfg.extra_top_k)["column"].tolist()
    elif cfg.use_extra_mode == "explicit":
        if not cfg.extra_explicit_cols:
            raise ValueError("Для use_extra_mode='explicit' нужно задать extra_explicit_cols")
        explicit_set = set(cfg.extra_explicit_cols)
        base_cols = [c for c in all_cols if c in explicit_set]
    else:
        raise ValueError(f"Unknown use_extra_mode: {cfg.use_extra_mode}")

    work = extra_stats[extra_stats["column"].isin(base_cols)].copy()

    if cfg.extra_filter_strategy in {"missing_only", "missing_and_variance", "score_topk"} and cfg.drop_extra_missing_over is not None:
        work = work[work["missing_rate"] <= cfg.drop_extra_missing_over]

    if cfg.extra_filter_strategy in {"variance_only", "missing_and_variance"}:
        work = work[work["std"].fillna(0.0) > cfg.variance_threshold]

    if cfg.extra_filter_strategy == "score_topk" and cfg.max_extra_keep_after_filter is not None:
        work = work.sort_values("quality_score", ascending=False).head(cfg.max_extra_keep_after_filter)

    cols = work["column"].tolist()
    return cols


def fit_extra_reducer(
    train_extra_pd: pd.DataFrame,
    cfg: ExperimentConfig,
) -> Dict:
    cols = [c for c in train_extra_pd.columns if c != "customer_id"]
    if not cols:
        return {
            "method": "none",
            "cols": [],
            "imputer": None,
            "reducer": None,
            "out_cols": [],
        }

    if cfg.extra_reduce_method == "none":
        return {
            "method": "none",
            "cols": cols,
            "imputer": None,
            "reducer": None,
            "out_cols": cols,
        }

    n_components = cfg.extra_reduce_dim
    if n_components is None:
        raise ValueError("Для PCA/SVD нужно задать extra_reduce_dim")

    imp = SimpleImputer(strategy="median")
    X_train = imp.fit_transform(train_extra_pd[cols]).astype(np.float32)

    n_components = min(n_components, X_train.shape[0], X_train.shape[1])
    if n_components < 1:
        raise ValueError("extra_reduce_dim слишком мал после проверки размеров")

    if cfg.extra_reduce_method == "pca":
        reducer = PCA(n_components=n_components, random_state=cfg.random_state)
        Z_train = reducer.fit_transform(X_train)
        out_cols = [f"extra_pca_{i}" for i in range(Z_train.shape[1])]
    elif cfg.extra_reduce_method == "svd":
        reducer = TruncatedSVD(n_components=n_components, random_state=cfg.random_state)
        Z_train = reducer.fit_transform(X_train)
        out_cols = [f"extra_svd_{i}" for i in range(Z_train.shape[1])]
    else:
        raise ValueError(f"Unknown extra_reduce_method: {cfg.extra_reduce_method}")

    return {
        "method": cfg.extra_reduce_method,
        "cols": cols,
        "imputer": imp,
        "reducer": reducer,
        "out_cols": out_cols,
    }


def transform_extra_with_reducer(extra_pd: pd.DataFrame, reducer_state: Dict) -> pd.DataFrame:
    method = reducer_state["method"]
    cols = reducer_state["cols"]

    if not cols:
        return extra_pd[["customer_id"]].copy()

    if method == "none":
        out = extra_pd[["customer_id"] + cols].copy()
        return reduce_mem_float32(out, exclude_cols=["customer_id"])

    X = reducer_state["imputer"].transform(extra_pd[cols]).astype(np.float32)
    Z = reducer_state["reducer"].transform(X).astype(np.float32)

    out = pd.concat(
        [
            extra_pd[["customer_id"]].reset_index(drop=True),
            pd.DataFrame(Z, columns=reducer_state["out_cols"]),
        ],
        axis=1
    )
    return out


def fit_feature_filter(
    train_df: pd.DataFrame,
    cat_cols: List[str],
    cfg: ExperimentConfig,
) -> Dict:
    feature_cols = [c for c in train_df.columns if c != "customer_id"]
    keep_cols = feature_cols.copy()
    removed = {
        "constant": [],
        "high_missing": [],
        "corr": [],
    }

    # 1) constant
    if cfg.feature_filter_strategy in {"drop_constant", "drop_constant_high_missing", "drop_constant_high_missing_corr"}:
        const_cols = []
        for c in keep_cols:
            nunique = train_df[c].nunique(dropna=False)
            if nunique <= 1:
                const_cols.append(c)
        removed["constant"] = const_cols
        keep_cols = [c for c in keep_cols if c not in set(const_cols)]

    # 2) high missing
    if cfg.feature_filter_strategy in {"drop_high_missing", "drop_constant_high_missing", "drop_constant_high_missing_corr"}:
        miss_cols = []
        for c in keep_cols:
            miss_rate = train_df[c].isna().mean()
            if miss_rate >= cfg.feature_missing_threshold:
                miss_cols.append(c)
        removed["high_missing"] = miss_cols
        keep_cols = [c for c in keep_cols if c not in set(miss_cols)]

    # 3) corr on numeric sample only
    if cfg.feature_filter_strategy == "drop_constant_high_missing_corr":
        num_cols = [c for c in keep_cols if c not in cat_cols]
        if len(num_cols) >= 2:
            sample_n = min(cfg.corr_sample_rows, len(train_df))
            sample_pd = train_df[num_cols].sample(sample_n, random_state=cfg.random_state).copy()

            # median fill only for corr estimation
            for c in num_cols:
                med = sample_pd[c].median()
                sample_pd[c] = sample_pd[c].fillna(med)

            sample_pd = reduce_mem_float32(sample_pd)
            corr = sample_pd.corr().abs()
            upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            to_drop_corr = [c for c in upper.columns if (upper[c] > cfg.corr_threshold).any()]
            removed["corr"] = to_drop_corr
            keep_cols = [c for c in keep_cols if c not in set(to_drop_corr)]

    return {
        "keep_cols": keep_cols,
        "removed": removed,
    }


def apply_feature_filter(df: pd.DataFrame, filter_state: Dict) -> pd.DataFrame:
    cols = ["customer_id"] + filter_state["keep_cols"]
    existing = [c for c in cols if c in df.columns]
    return df[existing].copy()


def apply_missing_strategy(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    cat_cols: List[str],
    cfg: ExperimentConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    train_df = train_df.copy()
    valid_df = valid_df.copy()

    feature_cols = [c for c in train_df.columns if c != "customer_id"]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    def add_indicators(base_df: pd.DataFrame, miss_cols: List[str], num_cols_local: List[str]) -> pd.DataFrame:
        for c in miss_cols:
            base_df[f"isna_{c}"] = base_df[c].isna().astype(np.int8)
        base_df["num_missing_total"] = base_df[num_cols_local].isna().sum(axis=1).astype(np.int32)
        return base_df

    strategy = cfg.missing_strategy

    if strategy == "native_catboost":
        final_cat_cols = [c for c in cat_cols if c in train_df.columns]
        return train_df, valid_df, final_cat_cols

    if strategy in {"median_plus_ind", "zero_plus_ind"}:
        miss_cols = [c for c in num_cols if train_df[c].isna().mean() > cfg.indicator_missing_threshold]
        train_df = add_indicators(train_df, miss_cols, num_cols)
        valid_df = add_indicators(valid_df, miss_cols, num_cols)

    if strategy in {"median", "median_plus_ind"}:
        imp = SimpleImputer(strategy="median")
        train_df[num_cols] = imp.fit_transform(train_df[num_cols]).astype(np.float32)
        valid_df[num_cols] = imp.transform(valid_df[num_cols]).astype(np.float32)
    elif strategy in {"zero", "zero_plus_ind"}:
        train_df[num_cols] = train_df[num_cols].fillna(0.0).astype(np.float32)
        valid_df[num_cols] = valid_df[num_cols].fillna(0.0).astype(np.float32)
    else:
        raise ValueError(f"Unknown missing_strategy: {strategy}")

    train_df = reduce_mem_float32(train_df, exclude_cols=cat_cols + ["customer_id"])
    valid_df = reduce_mem_float32(valid_df, exclude_cols=cat_cols + ["customer_id"])
    final_cat_cols = [c for c in cat_cols if c in train_df.columns]
    return train_df, valid_df, final_cat_cols


def prepare_train_valid_matrices(
    paths: PathsConfig,
    cfg: ExperimentConfig,
    extra_stats: Optional[pd.DataFrame] = None,
):
    parts = load_split_parts(paths)

    train_main_train = parts["train_main_train"]
    train_main_val = parts["train_main_val"]
    train_target_train = parts["train_target_train"]
    train_target_val = parts["train_target_val"]

    target_cols = get_target_cols(train_target_train.columns)
    cat_cols_main = get_cat_cols(train_main_train.columns)

    train_main_train = cast_cat_features_pl(train_main_train, cat_cols_main, as_string=True)
    train_main_val = cast_cat_features_pl(train_main_val, cat_cols_main, as_string=True)

    X_train_pd = train_main_train.to_pandas()
    X_val_pd = train_main_val.to_pandas()

    # optional quick subsample for debugging
    y_train_pd = train_target_train.select(target_cols).to_pandas().astype(np.int8)
    y_val_pd = train_target_val.select(target_cols).to_pandas().astype(np.int8)

    if cfg.sample_rows is not None:
        X_train_pd, y_train_pd = maybe_subsample_by_split(X_train_pd, y_train_pd, cfg.sample_rows, cfg.random_state)

    selected_extra_cols = []
    reducer_state = None

    if cfg.use_extra_mode != "none":
        if parts["train_extra_train_path"] is None or parts["train_extra_val_path"] is None:
            raise ValueError("Extra split files не найдены, но experiment config требует extra")

        if extra_stats is None:
            extra_stats = compute_extra_stats(parts["train_extra_train_path"])

        selected_extra_cols = choose_extra_columns(extra_stats, cfg)
        print(f"[extra] selected columns before reducer: {len(selected_extra_cols)}")

        if selected_extra_cols:
            tr_extra_pl = load_extra_subset(parts["train_extra_train_path"], selected_extra_cols)
            va_extra_pl = load_extra_subset(parts["train_extra_val_path"], selected_extra_cols)

            tr_extra_pd = tr_extra_pl.to_pandas()
            va_extra_pd = va_extra_pl.to_pandas()

            tr_extra_pd = reduce_mem_float32(tr_extra_pd, exclude_cols=["customer_id"])
            va_extra_pd = reduce_mem_float32(va_extra_pd, exclude_cols=["customer_id"])

            reducer_state = fit_extra_reducer(tr_extra_pd, cfg)
            tr_extra_final = transform_extra_with_reducer(tr_extra_pd, reducer_state)
            va_extra_final = transform_extra_with_reducer(va_extra_pd, reducer_state)

            X_train_pd = X_train_pd.merge(tr_extra_final, on="customer_id", how="left")
            X_val_pd = X_val_pd.merge(va_extra_final, on="customer_id", how="left")

            del tr_extra_pl, va_extra_pl, tr_extra_pd, va_extra_pd, tr_extra_final, va_extra_final
            gc.collect()

    # feature filter fit only on train
    filter_state = fit_feature_filter(X_train_pd, cat_cols_main, cfg)
    X_train_pd = apply_feature_filter(X_train_pd, filter_state)
    X_val_pd = apply_feature_filter(X_val_pd, filter_state)

    # missing strategy fit only on train
    X_train_pd, X_val_pd, final_cat_cols = apply_missing_strategy(
        train_df=X_train_pd,
        valid_df=X_val_pd,
        cat_cols=cat_cols_main,
        cfg=cfg,
    )

    return {
        "X_train": X_train_pd,
        "X_val": X_val_pd,
        "y_train": y_train_pd,
        "y_val": y_val_pd,
        "target_cols": target_cols,
        "cat_cols": final_cat_cols,
        "selected_extra_cols": selected_extra_cols,
        "reducer_state": reducer_state,
        "feature_filter_state": filter_state,
    }


def run_one_experiment(paths: PathsConfig, cfg: ExperimentConfig, extra_stats: Optional[pd.DataFrame] = None) -> Dict:
    exp_data = prepare_train_valid_matrices(paths, cfg, extra_stats=extra_stats)

    X_train = exp_data["X_train"]
    X_val = exp_data["X_val"]
    y_train = exp_data["y_train"]
    y_val = exp_data["y_val"]
    target_cols = exp_data["target_cols"]
    cat_cols = exp_data["cat_cols"]

    feature_cols = [c for c in X_train.columns if c != "customer_id"]

    train_pool = Pool(
        data=X_train[feature_cols],
        label=y_train[target_cols],
        cat_features=[c for c in cat_cols if c in feature_cols],
    )
    valid_pool = Pool(
        data=X_val[feature_cols],
        label=y_val[target_cols],
        cat_features=[c for c in cat_cols if c in feature_cols],
    )

    exp_name = (
        f"useextra={cfg.use_extra_mode}"
        f"__topk={cfg.extra_top_k}"
        f"__extrafilter={cfg.extra_filter_strategy}"
        f"__reduce={cfg.extra_reduce_method}_{cfg.extra_reduce_dim}"
        f"__featfilter={cfg.feature_filter_strategy}"
        f"__missing={cfg.missing_strategy}"
    )

    safe_mlflow_setup(cfg)

    if cfg.enable_mlflow and HAS_MLFLOW:
        with mlflow.start_run(run_name=exp_name):
            safe_mlflow_log_params(asdict(cfg))
            model = fit_catboost_model(
                train_pool=train_pool,
                valid_pool=valid_pool,
                cfg=cfg,
                random_seed=cfg.random_state,
                use_best_model=True,
            )
            val_pred = np.asarray(model.predict(valid_pool, prediction_type="Probability"), dtype=np.float32)
            metrics = multilabel_metrics(y_val[target_cols].values, val_pred)
            for k, v in metrics.items():
                mlflow.log_metric(k, v)
            mlflow.log_param("n_features", len(feature_cols))
            mlflow.log_param("n_cat_cols", len(cat_cols))
            mlflow.log_param("n_selected_extra_cols", len(exp_data["selected_extra_cols"]))
    else:
        model = fit_catboost_model(
            train_pool=train_pool,
            valid_pool=valid_pool,
            cfg=cfg,
            random_seed=cfg.random_state,
            use_best_model=True,
        )
        val_pred = np.asarray(model.predict(valid_pool, prediction_type="Probability"), dtype=np.float32)
        metrics = multilabel_metrics(y_val[target_cols].values, val_pred)

    result = {
        "experiment_name": exp_name,
        "macro_auc": metrics["macro_auc"],
        "macro_pr_auc": metrics["macro_pr_auc"],
        "mean_bce": metrics["mean_bce"],
        "hit_at_1": metrics["hit_at_1"],
        "n_features": len(feature_cols),
        "n_cat_cols": len(cat_cols),
        "n_selected_extra_cols": len(exp_data["selected_extra_cols"]),
        "selected_extra_cols_preview": exp_data["selected_extra_cols"][:15],
        "cfg": asdict(cfg),
        "feature_filter_removed": exp_data["feature_filter_state"]["removed"],
        "model": model,
    }

    del train_pool, valid_pool, X_train, X_val, y_train, y_val, val_pred
    gc.collect()

    return result


def make_experiment_grid(
    base_cfg: ExperimentConfig,
    use_extra_modes: List[str],
    extra_top_ks: List[Optional[int]],
    extra_filter_strategies: List[str],
    extra_reduce_methods: List[str],
    extra_reduce_dims: List[Optional[int]],
    feature_filter_strategies: List[str],
    missing_strategies: List[str],
) -> List[ExperimentConfig]:
    grid = []

    for use_extra_mode in use_extra_modes:
        topk_candidates = extra_top_ks if use_extra_mode == "topk" else [None]
        reduce_method_candidates = extra_reduce_methods if use_extra_mode != "none" else ["none"]

        for extra_top_k in topk_candidates:
            for extra_filter_strategy in extra_filter_strategies:
                for reduce_method in reduce_method_candidates:
                    dims = [None] if reduce_method == "none" else extra_reduce_dims
                    for reduce_dim in dims:
                        for feature_filter_strategy in feature_filter_strategies:
                            for missing_strategy in missing_strategies:
                                cfg = ExperimentConfig(**asdict(base_cfg))
                                cfg.use_extra_mode = use_extra_mode
                                cfg.extra_top_k = extra_top_k
                                cfg.extra_filter_strategy = extra_filter_strategy
                                cfg.extra_reduce_method = reduce_method
                                cfg.extra_reduce_dim = reduce_dim
                                cfg.feature_filter_strategy = feature_filter_strategy
                                cfg.missing_strategy = missing_strategy

                                # бессмысленные комбинации отсекаем
                                if use_extra_mode == "none" and reduce_method != "none":
                                    continue
                                if use_extra_mode == "none" and extra_filter_strategy != "none":
                                    continue
                                if use_extra_mode != "none" and reduce_method != "none" and reduce_dim is None:
                                    continue
                                if use_extra_mode != "topk" and extra_top_k is not None:
                                    continue

                                grid.append(cfg)
    return grid


def run_experiment_grid(
    paths: PathsConfig,
    experiment_grid: List[ExperimentConfig],
    artifacts_dir: str = "artifacts_fixed_val",
) -> pd.DataFrame:
    ensure_dir(artifacts_dir)

    parts = load_split_parts(paths)
    extra_stats = None
    if parts["train_extra_train_path"] is not None:
        print_header("COMPUTE EXTRA STATS ON TRAIN SPLIT ONCE")
        extra_stats = compute_extra_stats(parts["train_extra_train_path"])
        extra_stats.to_csv(Path(artifacts_dir) / "extra_train_stats.csv", index=False)

    results = []
    for i, cfg in enumerate(experiment_grid, start=1):
        print_header(f"EXPERIMENT {i}/{len(experiment_grid)}")
        print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))

        try:
            result = run_one_experiment(paths, cfg, extra_stats=extra_stats)
            results.append({k: v for k, v in result.items() if k != "model"})
        except Exception as e:
            results.append({
                "experiment_name": f"FAILED__{i}",
                "macro_auc": np.nan,
                "macro_pr_auc": np.nan,
                "mean_bce": np.nan,
                "hit_at_1": np.nan,
                "n_features": np.nan,
                "n_cat_cols": np.nan,
                "n_selected_extra_cols": np.nan,
                "selected_extra_cols_preview": [],
                "cfg": asdict(cfg),
                "feature_filter_removed": {},
                "error": str(e),
            })
            print(f"FAILED: {e}")

        gc.collect()

        results_df = pd.DataFrame(results)
        results_df.to_csv(Path(artifacts_dir) / "experiment_results_running.csv", index=False)

    results_df = pd.DataFrame(results).sort_values(
        ["macro_auc", "macro_pr_auc"],
        ascending=[False, False],
        na_position="last",
    ).reset_index(drop=True)

    results_df.to_csv(Path(artifacts_dir) / "experiment_results_final.csv", index=False)
    return results_df


def build_full_train_test_for_best_cfg(
    paths: PathsConfig,
    cfg: ExperimentConfig,
) -> Dict:
    # full labeled train
    train_main = pl.read_parquet(paths.train_main_path)
    train_target = pl.read_parquet(paths.train_target_path)
    test_main = pl.read_parquet(paths.test_main_path)

    target_cols = get_target_cols(train_target.columns)
    cat_cols_main = get_cat_cols(train_main.columns)

    train_main = cast_cat_features_pl(train_main, cat_cols_main, as_string=True)
    test_main = cast_cat_features_pl(test_main, cat_cols_main, as_string=True)

    X_train_pd = train_main.to_pandas()
    X_test_pd = test_main.to_pandas()
    y_pd = train_target.select(target_cols).to_pandas().astype(np.int8)

    selected_extra_cols = []
    reducer_state = None

    if cfg.use_extra_mode != "none" and os.path.exists(paths.train_extra_path) and os.path.exists(paths.test_extra_path):
        # важно: stats считаем по full train extra, так как финальная модель обучается на всем train
        tmp_parts = load_split_parts(paths)
        # для выбора колонок берем статистики train-split как proxy для reproducibility между экспериментами,
        # но reducer fit уже на полном train.
        extra_stats = compute_extra_stats(tmp_parts["train_extra_train_path"])
        selected_extra_cols = choose_extra_columns(extra_stats, cfg)

        if selected_extra_cols:
            train_extra = load_extra_subset(paths.train_extra_path, selected_extra_cols).to_pandas()
            test_extra = load_extra_subset(paths.test_extra_path, selected_extra_cols).to_pandas()

            train_extra = reduce_mem_float32(train_extra, exclude_cols=["customer_id"])
            test_extra = reduce_mem_float32(test_extra, exclude_cols=["customer_id"])

            reducer_state = fit_extra_reducer(train_extra, cfg)
            train_extra_final = transform_extra_with_reducer(train_extra, reducer_state)
            test_extra_final = transform_extra_with_reducer(test_extra, reducer_state)

            X_train_pd = X_train_pd.merge(train_extra_final, on="customer_id", how="left")
            X_test_pd = X_test_pd.merge(test_extra_final, on="customer_id", how="left")

            del train_extra, test_extra, train_extra_final, test_extra_final
            gc.collect()

    filter_state = fit_feature_filter(X_train_pd, cat_cols_main, cfg)
    X_train_pd = apply_feature_filter(X_train_pd, filter_state)
    X_test_pd = apply_feature_filter(X_test_pd, filter_state)

    # apply missing train->test
    feature_cols = [c for c in X_train_pd.columns if c != "customer_id"]
    num_cols = [c for c in feature_cols if c not in cat_cols_main]

    if cfg.missing_strategy == "native_catboost":
        final_cat_cols = [c for c in cat_cols_main if c in X_train_pd.columns]
    else:
        X_tr = X_train_pd.copy()
        X_te = X_test_pd.copy()

        def add_indicators(base_df: pd.DataFrame, miss_cols: List[str], num_cols_local: List[str]):
            for c in miss_cols:
                base_df[f"isna_{c}"] = base_df[c].isna().astype(np.int8)
            base_df["num_missing_total"] = base_df[num_cols_local].isna().sum(axis=1).astype(np.int32)
            return base_df

        if cfg.missing_strategy in {"median_plus_ind", "zero_plus_ind"}:
            miss_cols = [c for c in num_cols if X_tr[c].isna().mean() > cfg.indicator_missing_threshold]
            X_tr = add_indicators(X_tr, miss_cols, num_cols)
            X_te = add_indicators(X_te, miss_cols, num_cols)

        if cfg.missing_strategy in {"median", "median_plus_ind"}:
            imp = SimpleImputer(strategy="median")
            X_tr[num_cols] = imp.fit_transform(X_tr[num_cols]).astype(np.float32)
            X_te[num_cols] = imp.transform(X_te[num_cols]).astype(np.float32)
        elif cfg.missing_strategy in {"zero", "zero_plus_ind"}:
            X_tr[num_cols] = X_tr[num_cols].fillna(0.0).astype(np.float32)
            X_te[num_cols] = X_te[num_cols].fillna(0.0).astype(np.float32)

        X_train_pd = reduce_mem_float32(X_tr, exclude_cols=cat_cols_main + ["customer_id"])
        X_test_pd = reduce_mem_float32(X_te, exclude_cols=cat_cols_main + ["customer_id"])
        final_cat_cols = [c for c in cat_cols_main if c in X_train_pd.columns]

    return {
        "X_train": X_train_pd,
        "X_test": X_test_pd,
        "y_train": y_pd,
        "target_cols": target_cols,
        "cat_cols": final_cat_cols,
        "selected_extra_cols": selected_extra_cols,
        "feature_filter_state": filter_state,
        "reducer_state": reducer_state,
    }


def fit_full_and_predict_test(
    paths: PathsConfig,
    cfg: ExperimentConfig,
    output_path: str = "artifacts_fixed_val/final_submit.parquet",
):
    full_data = build_full_train_test_for_best_cfg(paths, cfg)

    X_train = full_data["X_train"]
    X_test = full_data["X_test"]
    y_train = full_data["y_train"]
    target_cols = full_data["target_cols"]
    cat_cols = full_data["cat_cols"]

    feature_cols = [c for c in X_train.columns if c != "customer_id"]

    train_pool = Pool(
        data=X_train[feature_cols],
        label=y_train[target_cols],
        cat_features=[c for c in cat_cols if c in feature_cols],
    )
    test_pool = Pool(
        data=X_test[feature_cols],
        cat_features=[c for c in cat_cols if c in feature_cols],
    )

    model = fit_catboost_model(
        train_pool=train_pool,
        valid_pool=None,
        cfg=cfg,
        random_seed=cfg.random_state,
        use_best_model=False,
    )

    test_pred = np.asarray(model.predict(test_pool, prediction_type="Probability"), dtype=np.float32)

    sample_submit = pl.read_parquet(paths.sample_submit_path)
    predict_cols = [c for c in sample_submit.columns if c != "customer_id"]
    expected_predict_cols = [c.replace("target_", "predict_") for c in target_cols]

    if predict_cols != expected_predict_cols:
        raise AssertionError(
            "Порядок predict-колонок в sample_submit не совпадает с mapping target_->predict_."
        )

    submit_pd = pd.concat(
        [
            X_test[["customer_id"]].reset_index(drop=True),
            pd.DataFrame(test_pred, columns=expected_predict_cols),
        ],
        axis=1,
    )
    submit_pd = submit_pd[sample_submit.columns]

    ensure_dir(str(Path(output_path).parent))
    pl.from_pandas(submit_pd).write_parquet(output_path)

    print_header("FINAL SUBMIT SAVED")
    print("output_path:", output_path)
    print("shape      :", submit_pd.shape)

    return submit_pd, model


In [ ]:
# ============================================================
# OPTIONAL: QUICK SANITY CHECKS AFTER SPLIT
# ============================================================

parts = load_split_parts(PATHS)

print("train_main_train:", parts["train_main_train"].shape)
print("train_main_val  :", parts["train_main_val"].shape)
print("train_target_train:", parts["train_target_train"].shape)
print("train_target_val  :", parts["train_target_val"].shape)
print("has extra split files:", parts["train_extra_train_path"] is not None)

_ = quick_missing_report_pl(parts["train_main_train"], "train_main_train", top_n=20)
_ = quick_target_eda(parts["train_target_train"])


## RUN ALL EDA EXPERIMENTS

Это **отдельная ячейка** для запуска всей сетки экспериментов.

Ниже показан разумный стартовый набор по best practices:

- `main only`
- `main + topk extra`
- `main + all extra`
- `extra`: без сжатия / `SVD` / `PCA`
- feature filtering:
  - `none`
  - `drop_constant_high_missing`
  - `drop_constant_high_missing_corr`
- missing:
  - `native_catboost`
  - `median_plus_ind`
  - `zero_plus_ind`

Совет по памяти:
- сначала гоняйте ограниченную сетку;
- потом расширяйте;
- для debug можно задать `BASE_CFG.sample_rows`.


In [ ]:
# ============================================================
# RUN ALL EDA EXPERIMENTS
# ============================================================

BASE_CFG = ExperimentConfig(
    random_state=42,
    sample_rows=None,  # например 100_000 для быстрых пробных прогонов

    # extra defaults
    use_extra_mode="none",
    extra_top_k=300,
    extra_filter_strategy="score_topk",
    drop_extra_missing_over=0.98,
    variance_threshold=0.0,
    max_extra_keep_after_filter=1000,
    extra_reduce_method="none",
    extra_reduce_dim=None,

    # merged feature filtering
    feature_filter_strategy="drop_constant_high_missing",
    feature_missing_threshold=0.98,
    corr_threshold=0.995,
    corr_sample_rows=30000,

    # missing
    missing_strategy="native_catboost",
    indicator_missing_threshold=0.95,

    # model
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=5.0,
    early_stopping_rounds=150,
    verbose_eval=200,

    task_type="GPU",   # если нет GPU, поменяйте на "CPU"
    devices="0",
    fallback_to_cpu_on_oom=True,
    gpu_ram_part=0.80,
    gpu_cat_features_storage="CpuPinnedMemory",
    max_ctr_complexity=1,
    border_count=64,
    thread_count=-1,

    # logging
    enable_mlflow=False,
    mlflow_tracking_uri="http://localhost:5000",
    mlflow_experiment_name="cyberpolka_fixed_val_experiments",
)

EXPERIMENT_GRID = make_experiment_grid(
    base_cfg=BASE_CFG,
    use_extra_modes=[
        "none",
        "topk",
        "all",
    ],
    extra_top_ks=[
        300,
        1000,
    ],
    extra_filter_strategies=[
        "none",
        "score_topk",
        "missing_and_variance",
    ],
    extra_reduce_methods=[
        "none",
        "svd",
        "pca",
    ],
    extra_reduce_dims=[
        32,
        64,
    ],
    feature_filter_strategies=[
        "none",
        "drop_constant_high_missing",
        "drop_constant_high_missing_corr",
    ],
    missing_strategies=[
        "native_catboost",
        "median_plus_ind",
        "zero_plus_ind",
    ],
)

print("n_experiments =", len(EXPERIMENT_GRID))

results_df = run_experiment_grid(
    paths=PATHS,
    experiment_grid=EXPERIMENT_GRID,
    artifacts_dir="artifacts_fixed_val",
)

results_df.head(20)


In [ ]:
# ============================================================
# INSPECT BEST CONFIGS
# ============================================================

pd.set_option("display.max_colwidth", 200)

best_cols = [
    "experiment_name",
    "macro_auc",
    "macro_pr_auc",
    "mean_bce",
    "hit_at_1",
    "n_features",
    "n_cat_cols",
    "n_selected_extra_cols",
]

display(results_df[best_cols].head(20))


## Финальное обучение на всем train и подготовка test-submit

Best practice после выбора лучшей конфигурации:
1. фиксируете лучший набор гиперпараметров и preprocessing из validation;
2. **переобучаете** пайплайн на **всем labeled train**;
3. тем же preprocessing трансформируете `test`;
4. сохраняете submit.

Ниже — готовая ячейка.


In [ ]:
# ============================================================
# FINAL TRAIN ON FULL LABELED TRAIN + PREDICT TEST
# ============================================================

# Пример: возьмем лучший эксперимент из results_df
best_cfg_dict = results_df.iloc[0]["cfg"]
if isinstance(best_cfg_dict, str):
    best_cfg_dict = json.loads(best_cfg_dict.replace("'", '"'))
BEST_CFG = ExperimentConfig(**best_cfg_dict)

final_submit_df, final_model = fit_full_and_predict_test(
    paths=PATHS,
    cfg=BEST_CFG,
    output_path="artifacts_fixed_val/final_submit.parquet",
)

final_submit_df.head()


## Что еще можно прогонять по best practices

Полезные дополнительные варианты экспериментов для этой задачи:

1. **Main-only baseline**
   - всегда нужен как референс;
   - без него сложно оценить, действительно ли extra помогает.

2. **Top-K extra vs all extra**
   - `topk=300`, `topk=1000`, `all`;
   - часто `all extra` проигрывает из-за шума и памяти.

3. **Reduction ablation**
   - `none`, `svd_32`, `svd_64`, `pca_32`, `pca_64`;
   - на очень разреженных/шумных extra SVD часто стабильнее PCA.

4. **Missingness ablation**
   - `native_catboost`
   - `median_plus_ind`
   - `zero_plus_ind`
   - индикаторы пропусков особенно полезны, когда пропуск — это сигнал.

5. **Feature filtering ablation**
   - только constant/high-missing
   - constant/high-missing + corr
   - без фильтрации
   - иногда CatBoost сам справляется, но correlation pruning может снять шум и ускорить fit.

6. **Final dataset creation**
   - после выбора лучшего конфига имеет смысл сохранить:
     - `best_config.json`
     - список выбранных extra колонок
     - список удаленных колонок
     - reducer params / explained variance
   - это помогает воспроизводимости команды.

7. **Практика для команды**
   - не менять split между участниками;
   - использовать единые имена run'ов;
   - хранить один `split_metadata.json`;
   - сравнивать только результаты на одном и том же validation.
